# QFF 2026 Phase 2 — PS-O1 (official benchmark)

Run all cells from the **Phase Two** working directory. This notebook regenerates the **official** 6-node QAOA and Processor D Bell evidence. The separate synthetic application does not replace the benchmark. All measurements below are Qiskit Aer simulator results, not hardware measurements.


## 1. Exact classical reference (64 assignments)
The official graph is fixed. Its verified optimum is 22; the random expected cut is 13.5. Check the imported helper is unchanged before submission.


In [1]:
from benchmark_phase2 import all_reference_values
optimum, optima, random_mean, random_opt_prob = all_reference_values()
assert optimum == 22 and random_mean == 13.5
print("Classical optimum:", optimum, "optimal assignments:", optima, "random mean:", random_mean)


Classical optimum: 22 optimal assignments: {(1, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 1)} random mean: 13.5


## 2. Optimize p=1 QAOA and measure processors A/B/D
Five Powell initialisations using ideal statevector expectation. Select the highest expected cut. The same angles are then used for B and D, under ideal and noisy conditions. 4096 shots per experiment. A is infeasible (5 physical vs 6 logical qubits). **This is a full rerun and may overwrite the previously saved simulator tables.**


In [2]:
from benchmark_phase2 import run
rows = run()
assert len(rows)==6


C*: 22 random mean: 13.5 optimal solutions: {(1, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 1)}
A ideal r=  P_opt= 
A noisy r=  P_opt= 
B ideal r= 0.7023481889204546 P_opt= 0.11865234375
B noisy r= 0.6953014026988636 P_opt= 0.105224609375
D ideal r= 0.7031693892045454 P_opt= 0.125244140625
D noisy r= 0.6961447975852273 P_opt= 0.113037109375
Five-start ideal statevector mean/std approximation ratio: 0.6772765900509509 0.022028157183727057
Wrote: C:\Users\Abheek\Downloads\QFF_Phase2_Audited_Ready_For_Laptop\Phase Two\results\tables\results.csv


## 3. Bell-state verification on Processor D
Adjacent and graph-diameter pairs are picked by the packaged selection algorithm. XX/YY/ZZ measured with 4096 shots each in ideal and noisy simulations; fidelity=(1+XX-YY+ZZ)/4.


In [3]:
from pathlib import Path
from qff_phase2 import load_processor
from bell_experiments import build_rows, write_csv
proc=load_processor(str(Path("processors/processor_D.json")))
bell_rows=build_rows(proc)
write_csv(bell_rows, Path("results/tables/bell.csv"))
print("Bell experiment rows:",len(bell_rows))


Bell experiment rows: 4


## 4. Figures, validation and evidence tables
Qiskit counts use reversed displayed bit order. `counts_to_x` maps them to vertex order `x0,...,x5` before scoring.


In [4]:
import subprocess,sys,csv
subprocess.run([sys.executable,"make_figures.py"],check=True)
subprocess.run([sys.executable,"plot_bell_results.py","results/tables/bell.csv","--out-dir","results/figures"],check=True)
subprocess.run([sys.executable,"check_outputs.py"],check=True)
for name in ["results.csv","resources.csv","bell.csv","qaoa_five_starts.csv"]:
    with open(Path("results/tables")/name,newline="") as f:
        data=list(csv.DictReader(f))
    print(name,len(data),"rows")
subprocess.run([sys.executable,"verify_release_evidence.py"],check=True)


results.csv 6 rows
resources.csv 3 rows
bell.csv 4 rows
qaoa_five_starts.csv 5 rows


CompletedProcess(args=['C:\\Users\\Abheek\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', 'verify_release_evidence.py'], returncode=0)

## 5. Interpretation and limits
The archived run showed D with 6 versus B with 9 physical qubits, and similar quality (noisy ratios ≈0.6961 versus ≈0.6953). Recomputed results may vary due to sampling and environment. The CX count was equal (21 each) and D had higher two-qubit depth (19 versus 17) in the archived run. **Do not infer statistically significant superiority, quantum advantage, hardware speedup, or real-world throughput.** Read `report.md` and `supplementary_mobile_network/README.md` for scope and limitations.
